# SQL in R Analysis

In [1]:
%load_ext rpy2.ipython

In [2]:
%%R

install.packages("sqldf")

library(sqldf)

Installing package into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)
also installing the dependencies ‘gsubfn’, ‘proto’, ‘RSQLite’, ‘chron’

trying URL 'https://cran.rstudio.com/src/contrib/gsubfn_0.7.tar.gz'
trying URL 'https://cran.rstudio.com/src/contrib/proto_1.0.0.tar.gz'
trying URL 'https://cran.rstudio.com/src/contrib/RSQLite_3.52.0.tar.gz'
trying URL 'https://cran.rstudio.com/src/contrib/chron_2.3-62.tar.gz'
trying URL 'https://cran.rstudio.com/src/contrib/sqldf_0.4-12.tar.gz'

The downloaded source packages are in
	‘/tmp/RtmpskOguS/downloaded_packages’
Loading required package: gsubfn
Loading required package: proto
Loading required package: RSQLite
In addition: Warning message:
no DISPLAY variable so Tk is not available 


In [3]:
%%R

deliveries <- read.csv("deliveries.csv")
orders <- read.csv("orders.csv")
complaints <- read.csv("complaints.csv")
drivers <- read.csv("drivers.csv")
hubs <- read.csv("hubs.csv")
incidents <- read.csv("incidents.csv")
vehicles <- read.csv("vehicles.csv")
customers <- read.csv("customers.csv")
app_events <- read.csv("app_events.csv")

cat("Datasets loaded successfully\n")

cat("Deliveries rows:", nrow(deliveries), "\n")
cat("Orders rows:", nrow(orders), "\n")
cat("Complaints rows:", nrow(complaints), "\n")
cat("Drivers rows:", nrow(drivers), "\n")
cat("Hubs rows:", nrow(hubs), "\n")
cat("Incidents rows:", nrow(incidents), "\n")
cat("Vehicles rows:", nrow(vehicles), "\n")
cat("Customers rows:", nrow(customers), "\n")
cat("App Events rows:", nrow(app_events), "\n")

Datasets loaded successfully
Deliveries rows: 950 
Orders rows: 1250 
Complaints rows: 320 
Drivers rows: 170 
Hubs rows: 8 
Incidents rows: 280 
Vehicles rows: 120 
Customers rows: 650 
App Events rows: 640 


## Query 1: Delivery Status Distribution

In [5]:
%%R

delivery_status_analysis <- sqldf("
SELECT
    delivery_status,
    COUNT(*) AS total_deliveries
FROM deliveries
GROUP BY delivery_status
ORDER BY total_deliveries DESC
")

print(delivery_status_analysis)

  delivery_status total_deliveries
1          OnTime              616
2         Delayed              202
3          Failed              132


Delivery status analysis shows that most deliveries were completed on time, but there are still 202 delayed deliveries and 132 failed deliveries. This shows that NorthStar has a clear service reliability issue, because a noticeable number of deliveries are not being completed as expected.

## Query 2: Average Fuel or Charge Cost by Hub

In [6]:
%%R

hub_cost_analysis <- sqldf("
SELECT
    h.zone,
    h.hub_name,
    h.hub_type,
    ROUND(AVG(d.fuel_or_charge_cost),2) AS avg_cost,
    COUNT(*) AS total_deliveries
FROM deliveries d
JOIN hubs h
ON d.hub_id = h.hub_id
GROUP BY h.zone, h.hub_name, h.hub_type
ORDER BY avg_cost DESC
")

print(hub_cost_analysis)

       zone       hub_name  hub_type avg_cost total_deliveries
1   Central   Central Core   Control    13.69              115
2   Airport    Airport Hub  Dispatch    13.32              104
3      West      West Gate  Dispatch    13.17              127
4 Riverside  Riverside Hub Warehouse    12.92              115
5     North North Exchange  Dispatch    12.76              136
6      East      East Dock Warehouse    12.74              119
7     South     South Link  Dispatch    12.57              106
8   Central  Midtown Relay  Charging    11.71              128


Average fuel or charging cost differs across hubs and zones. Hubs with higher average cost may be affected by longer routes, inefficient dispatch planning, congestion, or heavier operational workload. This supports the case study issue that some locations perform worse than others.


## Query 3: Complaint Severity Analysis

In [7]:
%%R

complaint_analysis <- sqldf("
SELECT
    severity,
    COUNT(*) AS total_complaints,
    ROUND(AVG(compensation_amount),2) AS avg_compensation
FROM complaints
GROUP BY severity
ORDER BY total_complaints DESC
")

print(complaint_analysis)

  severity total_complaints avg_compensation
1   Medium              172            17.37
2     High               77            38.85
3      Low               71             9.06


Complaint analysis shows that higher severity complaints are associated with larger compensation amounts and may indicate major service failures. Frequent complaints can negatively affect customer satisfaction and operational reputation.

## Query 4: Driver Manual Route Override Analysis

In [14]:
%%R

route_override_analysis <- sqldf("
SELECT
    driver_id,
    COUNT(*) AS total_deliveries,
    SUM(manual_route_override_count) AS total_route_overrides,
    ROUND(AVG(manual_route_override_count),2) AS avg_route_override
FROM deliveries
GROUP BY driver_id
HAVING total_deliveries >= 2
ORDER BY total_route_overrides DESC, avg_route_override DESC
LIMIT 10
")

print(route_override_analysis)

   driver_id total_deliveries total_route_overrides avg_route_override
1       D127                6                    17               2.83
2       D130                8                    16               2.00
3       D087               12                    16               1.33
4       D131                9                    15               1.67
5       D108               11                    15               1.36
6       D069                7                    14               2.00
7       D105                7                    14               2.00
8       D028                7                    13               1.86
9       D017               10                    13               1.30
10      D104                7                    12               1.71


The driver route override analysis was conducted to investigate whether drivers frequently modified system-generated routes during delivery operations. The SQL query analysed delivery records and calculated the number of manual overrides and average override behaviour associated with individual drivers.

## Query 5: Delayed Deliveries by Zone




In [13]:
%%R

delayed_dropoff_zone_analysis <- sqldf("
SELECT
    CASE
        WHEN UPPER(o.dropoff_zone) IN ('CENTRAL', 'CTR') THEN 'Central'
        WHEN UPPER(o.dropoff_zone) = 'AIRPORT' THEN 'Airport'
        WHEN UPPER(o.dropoff_zone) = 'NORTH' THEN 'North'
        WHEN UPPER(o.dropoff_zone) = 'SOUTH' THEN 'South'
        WHEN UPPER(o.dropoff_zone) = 'EAST' THEN 'East'
        WHEN UPPER(o.dropoff_zone) = 'WEST' THEN 'West'
        WHEN UPPER(o.dropoff_zone) = 'RIVERSIDE' THEN 'Riverside'
        ELSE o.dropoff_zone
    END AS cleaned_dropoff_zone,
    COUNT(*) AS delayed_deliveries
FROM deliveries d
JOIN orders o
ON d.order_id = o.order_id
WHERE d.delivery_status = 'Delayed'
GROUP BY cleaned_dropoff_zone
ORDER BY delayed_deliveries DESC
")

print(delayed_dropoff_zone_analysis)

  cleaned_dropoff_zone delayed_deliveries
1              Central                 36
2              Airport                 35
3                North                 32
4                 West                 26
5            Riverside                 26
6                South                 25
7                 East                 22


This analysis was conducted to identify whether certain drop-off zones experience higher levels of delivery delays than others. The results help reveal operational areas that may be underperforming due to traffic congestion, inefficient routing, or dispatch coordination problems. Identifying high-delay zones allows NorthStar to prioritise operational improvements within areas experiencing the greatest service disruption.

## Query 6: Delivery Status by Service Type

In [10]:
%%R

service_status_analysis <- sqldf("
SELECT
    o.service_type,
    d.delivery_status,
    COUNT(*) AS total_deliveries
FROM deliveries d
JOIN orders o
ON d.order_id = o.order_id
GROUP BY o.service_type, d.delivery_status
ORDER BY o.service_type, total_deliveries DESC
")

print(service_status_analysis)

   service_type delivery_status total_deliveries
1      Business          OnTime               73
2      Business         Delayed               28
3      Business          Failed               25
4       Medical          OnTime               70
5       Medical         Delayed               22
6       Medical          Failed               16
7        Parcel          OnTime              156
8        Parcel         Delayed               49
9        Parcel          Failed               25
10    Passenger          OnTime              171
11    Passenger         Delayed               53
12    Passenger          Failed               38
13       Retail          OnTime              146
14       Retail         Delayed               50
15       Retail          Failed               28


This query analyses whether delivery performance differs across NorthStar’s various service types. The SQL query joins the deliveries and orders datasets and groups records according to service type and delivery status to identify operational performance patterns across different services.

## Query 7: Complaint Severity by Service Type

In [11]:
%%R

complaint_service_analysis <- sqldf("
SELECT
    o.service_type,
    c.severity,
    COUNT(*) AS total_complaints,
    ROUND(AVG(c.compensation_amount),2) AS avg_compensation
FROM complaints c
JOIN orders o
ON c.order_id = o.order_id
GROUP BY o.service_type, c.severity
ORDER BY o.service_type, total_complaints DESC
")

print(complaint_service_analysis)

   service_type severity total_complaints avg_compensation
1      Business   Medium               23            18.05
2      Business     High                9            37.50
3      Business      Low                7             8.30
4       Medical   Medium               17            14.99
5       Medical     High               11            40.84
6       Medical      Low                9             8.53
7        Parcel   Medium               36            16.44
8        Parcel     High               22            37.16
9        Parcel      Low               19            12.42
10    Passenger   Medium               45            17.18
11    Passenger     High               22            39.45
12    Passenger      Low               17             7.63
13       Retail   Medium               51            18.67
14       Retail      Low               19             7.40
15       Retail     High               13            39.82


This query analyses complaint severity levels across different NorthStar service types to identify which services generate the highest operational and customer service issues.

## Query 8: Average Customer Rating by Delivery Status

In [12]:
%%R

rating_status_analysis <- sqldf("
SELECT
    delivery_status,
    ROUND(AVG(customer_rating_post_delivery),2) AS avg_customer_rating,
    COUNT(*) AS total_deliveries
FROM deliveries
WHERE customer_rating_post_delivery IS NOT NULL
GROUP BY delivery_status
ORDER BY avg_customer_rating DESC
")

print(rating_status_analysis)

  delivery_status avg_customer_rating total_deliveries
1          OnTime                4.28              608
2         Delayed                3.11              197
3          Failed                3.05              131


This query analyses how delivery performance affects customer satisfaction by comparing average customer ratings across different delivery status categories.